In [1]:
# ============================================================
# E4 Wristband Sleep Staging v3.1 — SAME architecture as v3
# (Triple-CNN + BiMamba + SupCon, unified 3-channel input),
# ONLY CHANGE: removed EEG-borrowed artifact weighting.
#
# This is a clean A/B test against the original v3 run:
#   v3   : 48.75 ± 1.40% Acc, F1 0.4624, Kappa 0.2988  (EEG-weighted)
#   v3.1 : this script                                  (uniform weight)
#
# Uses the SAME preprocessed data as v3 (preprocessed_E4_v3_BHT) —
# no need to re-run preprocessing.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS & CONFIG — identical to v3 except EVAL_PATH
# ============================================================
E4_PATH   = r"D:\22\AA\preprocess\preprocessed_E4_v3_BHT"    # reuse v3 data
EVAL_PATH = r"D:\22\AA\evaluation\e4_v3_1_no_eeg_weight"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES   = ["Wake", "N1", "N2", "N3", "REM"]
CONTEXT       = 7
WINDOW        = 2 * CONTEXT + 1
N_EPOCHS      = 30
BATCH_SIZE    = 64
SEEDS         = [42, 123, 256, 789, 999]
IN_CH         = 3
EPOCH_SAMPLES = 1920   # 64Hz x 30s (same as v3)

D_MODEL       = 128    # same as v3 (isolating the artifact-weight variable only)
D_STATE       = 16
DROPOUT       = 0.4
SUPCON_LAMBDA = 0.2
SUPCON_TEMP   = 0.05
PROJ_DIM      = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"Architecture  : Triple-CNN + BiMamba + SupCon (SAME as v3)")
print(f"Change        : EEG artifact weighting REMOVED -> using E4's own")
print(f"                flat/NaN quality mask instead (uniform otherwise)")
print(f"Output        : {EVAL_PATH}")

TRAIN_SUBS = np.load(os.path.join(E4_PATH, '_train_subs.npy'), allow_pickle=True).tolist()
TEST_SUBS  = np.load(os.path.join(E4_PATH, '_test_subs.npy'),  allow_pickle=True).tolist()
print(f"Train: {len(TRAIN_SUBS)}  Test: {len(TEST_SUBS)}")


# ============================================================
# E4's OWN quality check (replaces EEG-borrowed artifact weights)
# Simple, modality-appropriate: flag epochs where BVP is flat/NaN
# (broken sensor contact), everything else gets weight 1.
# ============================================================
def e4_own_quality(e4_array):
    """e4_array: (N, 3, 1920) -> weights (N,) of 1.0 or 0.0"""
    n = e4_array.shape[0]
    weights = np.ones(n, dtype=np.float32)
    bvp = e4_array[:, 0, :]   # BVP is channel 0
    for i in range(n):
        sig = bvp[i]
        if np.isnan(sig).any() or sig.std() < 1e-6:
            weights[i] = 0.0
    return weights


# ============================================================
# DATASET — same structure as v3, weight source changed
# ============================================================
class E4Dataset(Dataset):
    def __init__(self, subject_list, data_path, context=CONTEXT):
        self.context = context
        self.data  = []
        self.index = []
        label_counter = Counter()

        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                continue
            with np.load(fp) as d:
                e4 = d['e4'].astype(np.float32)   # (N, 3, 1920)
                labels = d['labels'].copy()

            n = len(labels)
            art_w = e4_own_quality(e4)   # <-- ONLY functional change vs v3

            sub_idx = len(self.data)
            self.data.append((e4, labels, art_w))

            for i in range(n):
                self.index.append((sub_idx, i, n, float(art_w[i])))
                label_counter[int(labels[i])] += 1

        self.label_counts = np.array([label_counter[i] for i in range(5)], dtype=np.float32)
        total = len(self.index)
        ram = sum(e.nbytes for e, _, _ in self.data) / 1e9
        dropped = sum(1 for *_, w in self.index if w == 0)
        print(f"  Samples : {total:,}   RAM: {ram:.2f} GB   Flat/NaN dropped: {dropped} ({dropped/total*100:.2f}%)")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n, weight = self.index[idx]
        e4, labels, _ = self.data[sub_idx]

        window_epochs = []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            window_epochs.append(e4[ei])

        x = np.stack(window_epochs, axis=0)   # (W, 3, 1920)
        y = int(labels[center_i])
        return (
            torch.FloatTensor(x),
            torch.tensor(y, dtype=torch.long),
            torch.tensor(weight, dtype=torch.float32),
        )


print("\nBuilding datasets...")
train_ds = E4Dataset(TRAIN_SUBS, E4_PATH)
test_ds  = E4Dataset(TEST_SUBS,  E4_PATH)
print("Datasets ready.")


# ============================================================
# MODEL — IDENTICAL to v3 (TripleResCNN + BiMamba + SupCon)
# ============================================================
class TripleResCNN(nn.Module):
    def __init__(self, in_ch=3, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        mid = d_model // 3

        def branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=4, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=5, padding=2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = branch(7)
        self.medium = branch(13)
        self.large  = branch(25)

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, EPOCH_SAMPLES)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]

        target_L = min(L_s, L_m, L_l)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)

        self.proj = nn.Sequential(
            nn.Conv1d(3 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))
        return self.proj(torch.cat([fs, fm, fl], dim=1))


class MambaBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, d_state=D_STATE, d_conv=4, expand=2, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = d_model * expand

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d  = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                  padding=d_conv - 1, groups=self.d_inner, bias=True)
        self.x_proj  = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)
        self.A_log = nn.Parameter(torch.log(A.expand(self.d_inner, -1)))
        self.D     = nn.Parameter(torch.ones(self.d_inner))

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm     = nn.LayerNorm(d_model)
        self.dropout  = nn.Dropout(dropout)

    def ssm_scan(self, x, A, B, C, D):
        B_size, L, d = x.shape
        h = torch.zeros(B_size, d, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for i in range(L):
            xi = x[:, i, :]
            h = h * A.unsqueeze(0) + xi.unsqueeze(-1) * B[:, i, :].unsqueeze(1)
            y = (h * C[:, i, :].unsqueeze(1)).sum(-1) + D * xi
            ys.append(y)
        return torch.stack(ys, dim=1)

    def forward(self, x):
        residual = x
        B, L, _ = x.shape
        xz = self.in_proj(x)
        x_, z = xz.chunk(2, dim=-1)
        x_ = F.silu(self.conv1d(x_.transpose(1, 2))[:, :, :L].transpose(1, 2))
        ssm_p = self.x_proj(x_)
        dt_r, Bp, Cp = ssm_p.split([1, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_r))
        A = -torch.exp(self.A_log)
        A_disc = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0)).mean(1).mean(0)
        y = self.ssm_scan(x_, A_disc, Bp, Cp, self.D)
        y = self.out_proj(self.dropout(y * F.silu(z)))
        return self.norm(y + residual)


class BiMambaBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, d_state=D_STATE, dropout=DROPOUT):
        super().__init__()
        self.fwd = MambaBlock(d_model, d_state, dropout=dropout)
        self.bwd = MambaBlock(d_model, d_state, dropout=dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        fwd = self.fwd(x)
        bwd = torch.flip(self.bwd(torch.flip(x, [1])), [1])
        return self.norm(fwd + bwd)


class E4BitMamSleep(nn.Module):
    def __init__(self, in_ch=IN_CH, d_model=D_MODEL, d_state=D_STATE, n_layers=2, dropout=DROPOUT,
                 n_classes=5, context=CONTEXT, proj_dim=PROJ_DIM):
        super().__init__()
        self.context = context
        self.d_model = d_model

        self.triple_cnn    = TripleResCNN(in_ch, d_model, dropout)
        self.intra_bimamba = nn.Sequential(*[BiMambaBlock(d_model, d_state, dropout) for _ in range(n_layers)])
        self.inter_pos     = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.01)
        self.inter_bimamba = nn.Sequential(*[BiMambaBlock(d_model, d_state, dropout) for _ in range(n_layers)])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes),
        )
        self.projector = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model), nn.GELU(),
            nn.Linear(d_model, proj_dim),
        )

    def forward(self, x):
        B, W, C, T = x.shape
        cnn_out = self.triple_cnn(x.view(B * W, C, T)).permute(0, 2, 1)
        intra = self.intra_bimamba(cnn_out).mean(dim=1).view(B, W, self.d_model)
        inter = self.inter_bimamba(intra + self.inter_pos)
        center = inter[:, self.context, :]
        logits = self.classifier(center)
        proj = F.normalize(self.projector(center), dim=1)
        return logits, proj


# ============================================================
# LOSS — identical to v3
# ============================================================
class SupConLoss(nn.Module):
    def __init__(self, temperature=SUPCON_TEMP):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        B = features.shape[0]
        dev = features.device
        sim = torch.matmul(features, features.T) / self.temperature
        sim = sim - sim.max(dim=1, keepdim=True)[0].detach()
        pos = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
        sm = torch.eye(B, device=dev)
        pos = pos - sm
        denom = torch.exp(sim) * (1 - sm)
        lp = sim - torch.log(denom.sum(1, keepdim=True) + 1e-8)
        n_pos = pos.sum(1)
        am = (n_pos > 0).float()
        loss = -(pos * lp).sum(1) / (n_pos + 1e-8)
        return (loss * am).sum() / (am.sum() + 1e-8)


supcon_loss = SupConLoss()
cw_np = train_ds.label_counts.sum() / (5 * train_ds.label_counts)
cw = torch.FloatTensor(cw_np).to(device)
print(f"\nClass weights: " + " | ".join(f"{n}:{w:.2f}" for n, w in zip(LABEL_NAMES, cw_np)))


def combined_loss(logits, proj, labels, weights):
    ce = nn.CrossEntropyLoss(weight=cw, reduction='none')(logits, labels)
    wce = ce * weights
    valid = weights > 0
    l_ce = wce[valid].mean() if valid.sum() > 0 else wce.mean()
    l_sc = supcon_loss(proj, labels)
    return l_ce + SUPCON_LAMBDA * l_sc, l_ce.item(), l_sc.item()


# ============================================================
# HELPERS — identical to v3
# ============================================================
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    tl = tc = ts = 0
    preds, labs = [], []
    for x, y, w in loader:
        x, y, w = x.to(device), y.to(device), w.to(device)
        optimizer.zero_grad()
        logits, proj = model(x)
        loss, lc, ls = combined_loss(logits, proj, y, w)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        tl += loss.item(); tc += lc; ts += ls
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(loader)
    return (tl/n, tc/n, ts/n, accuracy_score(labs, preds),
            f1_score(labs, preds, average='macro', zero_division=0))


def evaluate(model, loader):
    model.eval()
    preds, labs = [], []
    with torch.no_grad():
        for x, y, w in loader:
            logits, _ = model(x.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            labs.extend(y.numpy())
    preds, labs = np.array(preds), np.array(labs)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs, preds)
    per = f1_score(labs, preds, average=None, zero_division=0)
    return acc, f1, kappa, per


# ============================================================
# CSV
# ============================================================
csv_summary = os.path.join(EVAL_PATH, "summary.csv")
fields = ["seed", "acc", "f1_macro", "kappa", "f1_Wake", "f1_N1", "f1_N2", "f1_N3", "f1_REM"]
with open(csv_summary, 'w', newline='') as f:
    csv.DictWriter(f, fields).writeheader()


# ============================================================
# 5-SEED TRAINING
# ============================================================
all_results = []
for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=0, generator=torch.Generator().manual_seed(seed))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = E4BitMamSleep().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}  (same as v3: ~991K)")

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4, betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def lr_fn(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_fn)
    best_f1, best_path = 0.0, os.path.join(EVAL_PATH, f"best_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tl, tc, ts, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler)
        acc, f1, kap, per = evaluate(model, test_loader)

        saved = ""
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), best_path)
            saved = " <- BEST"

        lr = optimizer.param_groups[0]['lr']
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tl:.3f}(CE:{tc:.3f}+SC:{ts:.3f}) "
              f"TrAcc:{tr_acc:.3f} ValAcc:{acc:.3f} F1:{f1:.3f} k:{kap:.3f} LR:{lr:.2e}{saved}")

        if epoch % 10 == 0:
            print("    Per-class: " + " | ".join(f"{LABEL_NAMES[i]}={per[i]:.3f}" for i in range(5)))

    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate(model, test_loader)

    print(f"\n  Seed {seed} FINAL: Acc={fin_acc*100:.2f}%  F1={fin_f1:.4f}  Kappa={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap, 'per_cls': fin_per})

    with open(csv_summary, 'a', newline='') as f_:
        csv.DictWriter(f_, fields).writerow({
            "seed": seed, "acc": round(fin_acc, 4), "f1_macro": round(fin_f1, 4), "kappa": round(fin_kap, 4),
            "f1_Wake": round(fin_per[0], 4), "f1_N1": round(fin_per[1], 4),
            "f1_N2": round(fin_per[2], 4), "f1_N3": round(fin_per[3], 4), "f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs = np.array([r['acc'] for r in all_results]) * 100
f1s = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])

print(f"\n{'='*60}\n5-SEED FINAL REPORT — E4 v3.1 (no EEG artifact weight)\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} ± {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} ± {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} ± {kappas.std():.4f}")
print(f"\nA/B COMPARISON (same architecture, only weighting source differs):")
print(f"  v3   (EEG-weighted)   : Acc=48.75±1.40%  F1=0.4624±0.0053  Kappa=0.2988±0.0075")
print(f"  v3.1 (own/no weight)  : Acc={accs.mean():.2f}±{accs.std():.2f}%  F1={f1s.mean():.4f}±{f1s.std():.4f}  Kappa={kappas.mean():.4f}±{kappas.std():.4f}")
print(f"\nSummary saved: {csv_summary}")

Device        : cuda
Architecture  : Triple-CNN + BiMamba + SupCon (SAME as v3)
Change        : EEG artifact weighting REMOVED -> using E4's own
                flat/NaN quality mask instead (uniform otherwise)
Output        : D:\22\AA\evaluation\e4_v3_1_no_eeg_weight
Train: 73  Test: 19

Building datasets...
  Samples : 68,857   RAM: 1.59 GB   Flat/NaN dropped: 78 (0.11%)
  Samples : 18,899   RAM: 0.44 GB   Flat/NaN dropped: 0 (0.00%)
Datasets ready.

Class weights: Wake:1.84 | N1:3.14 | N2:0.45 | N3:1.00 | REM:1.12

SEED 42  (1/5)
  Parameters: 991,419  (same as v3: ~991K)
  Ep[01/30] Loss:2.406(CE:1.435+SC:4.851) TrAcc:0.367 ValAcc:0.377 F1:0.367 k:0.204 LR:3.33e-04 <- BEST
  Ep[02/30] Loss:2.065(CE:1.251+SC:4.066) TrAcc:0.429 ValAcc:0.429 F1:0.416 k:0.247 LR:5.00e-04 <- BEST
  Ep[03/30] Loss:1.913(CE:1.111+SC:4.010) TrAcc:0.486 ValAcc:0.415 F1:0.427 k:0.262 LR:4.97e-04 <- BEST
  Ep[04/30] Loss:1.789(CE:1.001+SC:3.944) TrAcc:0.535 ValAcc:0.450 F1:0.455 k:0.289 LR:4.91e-04 <- BEST
  